In [29]:
import torch.nn as nn
import pandas as pd
import json
import numpy as np

In [30]:
class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_dim: int = 1,
        hidden_dim: int = 64,
        latent_dim: int = 32,
        num_layers: int = 1,
    ):
        
        super().__init__()
        
        # ---- Encoder ---- #
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        
        # ---- Latent ---- #
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)
        
        # ---- Decoder ---- #
        self.decoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        
    # ---- Forward ---- #
    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        hidden = hidden[-1]
        latent = self.to_latent(hidden)
        hidden_dec = self.from_latent(latent)
        seq_len = x.size(1)
        repeated_hidden = hidden_dec.unsqueeze(1).repeat(1, seq_len, 1)
        reconstructed, _ = self.decoder(repeated_hidden)
        return reconstructed

In [36]:
def load_series(csv_path: str):
    df = pd.read_csv(csv_path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df

In [ ]:
import json

def load_windows(json_path: str, file_key: str):
    with open(json_path, "r") as f:
        data = json.load(f)
    return data[file_key]

[['2014-04-10 07:15:00.000000', '2014-04-11 16:45:00.000000']]

In [34]:
def build_labels(df, windows):
    labels = np.zeros(len(df))
    for start, end in windows:
        start = pd.to_datetime(start)
        end = pd.to_datetime(end)
        
        mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)
        labels[mask] = 1

    df["labels"] = labels
    return df

In [35]:
def make_windows(values, seq_len):
    windows = []

    for i in range(len(values) - seq_len):
        window = values[i : i + seq_len]
        windows.append(window)

    return np.array(windows)

In [ ]:
SEQ_LEN = 30

df = load_series("../assets/data/art_daily_flatmiddle.csv")
windows = load_windows(
    json_path="../assets/labels/combined_windows.json",
    file_key="artificialWithAnomaly/art_daily_flatmiddle.csv"
)

df_labeled = build_labels(df, windows)

values = df_labeled["value"].values
x = make_windows(values, SEQ_LEN)

[[-21.04838268 -20.29547687 -18.12722947 ... -18.53693332 -21.34795385
  -20.99901373]
 [-20.29547687 -18.12722947 -20.1716654  ... -21.34795385 -20.99901373
  -19.52936452]
 [-18.12722947 -20.1716654  -21.22376161 ... -20.99901373 -19.52936452
  -21.31394633]
 ...
 [-20.6567604  -19.33631218 -19.96890288 ... -18.67665505 -18.08356203
  -20.27840639]
 [-19.33631218 -19.96890288 -21.73314535 ... -18.08356203 -20.27840639
  -20.06323875]
 [-19.96890288 -21.73314535 -19.16426173 ... -20.27840639 -20.06323875
  -20.7519731 ]]
